In [4]:
!pip install -q -U ollama


In [5]:
import json
import time
import io
import re
import ollama
import pandas as pd

print("✅ 初期化完了：ローカルのOllama (gemma4:e4b) を使用します。")

def extract_demographics_dst(user):
    """テキストと実際の画像（ローカルから）を同時に処理し、年代と性別の質量を抽出する。"""
    
    handle = user.get('handle', '')
    bio = user.get('bio', '')
    posts_text = [post.get('text', '') for post in user.get('posts', [])[:3]]

    prompt = f"""
あなたはX（旧Twitter）ユーザーの属性推定を行う、慎重かつ公平なアノテーターです。
あなたのタスクは、各ユーザーのプロフィールやコンテンツから得られる属性の手がかりについて段階的に推論し、Dempster-Shafer Theory (DST) に基づく構造化されたデータを出力することです。

各ユーザーについて、以下の情報が提供されます：
・ユーザー名 / ハンドルネーム (Handle)
・表示名 (Display Name)
・短い自己紹介 (Bio)
・15つのポスト (Tweets)
・プロフィール画像 (Profile Image)

これらの入力に基づき、ユーザーの【年齢層】と【性別】を推定してください。

分析の観点：
・名前の形態論と文字・言語形式（接尾辞、命名規則など）
・言語スタイル、文法、語彙の選択
・ポストの話題、興味・関心、トーン
・文化的、地域的、または言語的な手がかり
・テキスト内の絵文字、記号、または文体的な特徴
・プロフィール画像から得られる年齢、性別に関する視覚的な手がかり

DST（Dempster-Shafer Theory）のルール：
証拠が不十分または曖昧な場合は、無理に推測せず、不確実性（Omega）に質量（mass）を割り当ててください。
各属性の質量関数の合計は必ず 1.00 になるようにしてください。

【年齢層】
A = 18–24歳
B = 25–34歳
C = 35–44歳
D = 45–54歳
E = 55–64歳
F = 65歳以上
ルール: m(A) + m(B) + m(C) + m(D) + m(E) + m(F) + m(Omega) = 1.00

【性別】
M = 男性
F = 女性
ルール: m(M) + m(F) + m(Omega) = 1.00

---
データ
Handle: {handle}
Display Name: {handle}
Bio: {bio}
Posts: {posts_text}
---

出力は必ず以下のJSONフォーマットのみにしてください。JSON以外のテキストは絶対に出力しないでください：
{{
 "reasoning": "年齢と性別の推論過程、および特定した具体的な証拠を日本語で説明してください",
 "mass_age": {{"18_24": 0.00, "25_34": 0.00, "35_44": 0.00, "45_54": 0.00, "55_64": 0.00, "65_plus": 0.00, "Omega": 1.00}},
 "mass_gender": {{"M": 0.00, "F": 0.00, "Omega": 1.00}}
}}
"""

    imagenes_reales = []
    profile_img_filename = user.get('profile_picture_url')
    if profile_img_filename:
        # load_local_image は別セルで定義されているため、ここではそのまま呼び出す
        try:
            profile_img = load_local_image(profile_img_filename)
            if profile_img:
                imagenes_reales.append(profile_img)
        except NameError:
            print("警告: load_local_image が未定義です。")

    for post in user.get('posts', []):
        img_filename = post.get('image_file')
        if pd.notna(img_filename) and str(img_filename).strip() not in ["", "N/A", "nan"]:
            try:
                img = load_local_image(img_filename)
                if img:
                    imagenes_reales.append(img)
            except NameError:
                pass

    images_bytes = []
    for img in imagenes_reales:
        try:
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='PNG')
            images_bytes.append(img_byte_arr.getvalue())
        except Exception as e:
            print(f"    [警告] 画像変換エラー: {e}")

    max_retries = 3
    use_images = True
    for attempt in range(max_retries):
        try:
            message = {'role': 'user', 'content': prompt}
            if use_images and images_bytes:
                message['images'] = images_bytes

            response = ollama.chat(
                model='gemma4:e4b',
                messages=[ 
                    {'role': 'system', 'content': 'あなたは優秀なアノテーターです。JSON形式で正確に出力してください。'},
                    message
                ],
                stream=False
            )
            
            response_text = response['message']['content']
            
            match = re.search(r'```(?:json)?\s*(.*?)\s*```', response_text, re.DOTALL)
            if match:
                response_text = match.group(1)
                
            return json.loads(response_text)

        except Exception as e:
            error_msg = str(e)
            print(f"  [Error] ローカルモデル推論失敗 ({error_msg})。 (試行 {attempt+1})")
            
            # 画像が非対応のモデルだった場合は、次回以降の試行でテキストのみにする
            if "images" in error_msg.lower() or "vision" in error_msg.lower() or "support" in error_msg.lower():
                print("    -> 画像未対応モデルの可能性があるため、次回からテキストのみで推論します。")
                use_images = False
            
            time.sleep(5)

            if attempt == max_retries - 1:
                return {
                    "reasoning": f"エラーのため推論不可 ({error_msg})",
                    "mass_age": {"18_24": 0.00, "25_34": 0.00, "35_44": 0.00, "45_54": 0.00, "55_64": 0.00, "65_plus": 0.00, "Omega": 1.00},
                    "mass_gender": {"M": 0.00, "F": 0.00, "Omega": 1.00}
                }

print("✅ ブロック2 (年代と性別予測のOllama・スマートリトライ) がロードされました。")


✅ 初期化完了：ローカルのOllama (gemma4:e4b) を使用します。
✅ ブロック2 (年代と性別予測のOllama・スマートリトライ) がロードされました。


In [ ]:
# Imports and Local Paths Setting
# ---------------------------------
import pandas as pd
import json
import numpy as np
import os
from PIL import Image
# from google.colab import drive  # ← ローカルでは不要なのでコメントアウト
from IPython.display import display
import time # Ensure time is imported here, as it's used later

# 1. Google Driveをマウントする処理を無効化
# drive.mount('/content/drive')  # ← コメントアウト

# 画像が保存されているローカルのフォルダパス（相対パス）に変更
IMAGE_DIR = './dataset_images/年齢性別推定用'

# Helper Function: load_local_image
# --------------------------------
def load_local_image(filename):
    """ファイル名を受け取り、ローカルから画像を読み込んでリサイズする"""
    if pd.isna(filename) or str(filename).strip() in ["", "N/A", "nan"]:
        return None

    file_path = os.path.join(IMAGE_DIR, str(filename).strip())

    try:
        img = Image.open(file_path)
        img.thumbnail((512, 512))
        return img
    except Exception as e:
        print(f"    [警告] 画像が見つからないか、開けませんでした: {filename}")
        return None

# Data Loading and Preprocessing
# -------------------------------
# CSVファイルのパスもローカル用に変更
csv_path = './age.csv'
df = pd.read_csv(csv_path, header=0)
df.columns = [str(col).strip() for col in df.columns]

def is_valid(val):
    """pandasの値が有効か（NaN、null、またはN/Aではないか）を確認する"""
    return pd.notna(val) and str(val).strip() not in ["", "N/A", "nan"]

users_data = []
for _, row in df.iterrows():
    posts = []
    for i in range(1, 15):
        if is_valid(row.get(f'post_{i}_text')):
            img_filename = str(row[f'post_{i}_image_url']) if is_valid(row.get(f'post_{i}_image_url')) else None
            posts.append({
                "text": str(row[f'post_{i}_text']),
                "image_file": img_filename
            })

    interactions = []
    for i in range(1, 15):
        if is_valid(row.get(f'interaction_{i}_bio')):
            interactions.append(str(row[f'interaction_{i}_bio']))

    users_data.append({
        "user_id": str(row['user_id']),
        "handle": str(row['handle']) if is_valid(row.get('handle')) else "",
        "bio": str(row['bio_description']) if is_valid(row.get('bio_description')) else "",
        "profile_picture_url": str(row['profile_picture_url']) if is_valid(row.get('profile_picture_url')) else None,
        "posts": posts,
        "interactions": interactions,
        "ground_truth": str(row.get('ground_truth', 'Unknown'))
    })

print(f"✅ {len(users_data)}人のユーザーが正常にロードされました。")
print("✅ ローカルデータの読み込み準備が完了しました。")

# Demographics Prediction Loop
# ----------------------------
print("推論を開始します...\n")

results = []
age_categories = {
    "18_24": "18-24",
    "25_34": "25-34",
    "35_44": "35-44",
    "45_54": "45-54",
    "55_64": "55-64",
    "65_plus": "65+"
}
gender_categories = {"M": "Male", "F": "Female"}

for user in users_data[:7]: # Limiting to 7 users for demonstration as per original
    print(f"ユーザー {user['user_id']} ({user['ground_truth']}) を処理中...")

    # extract_demographics_dst is defined in cell cqVsexLc4RfE
    llm_inference = extract_demographics_dst(user)
    mass_age = llm_inference['mass_age']
    mass_gender = llm_inference['mass_gender']
    reasoning = llm_inference['reasoning']

    # Age Prediction Logic
    if mass_age.get("Omega", 0) > 0.5:
        prediction_age = "不確実"
    else:
        max_age_mass = -1
        prediction_age_key = "不確実"
        for key, value in mass_age.items():
            if key != "Omega" and value > max_age_mass:
                max_age_mass = value
                prediction_age_key = key
        prediction_age = age_categories.get(prediction_age_key, "不確実")

    # Gender Prediction Logic
    if mass_gender.get("Omega", 0) > 0.5:
        prediction_gender = "不確実"
    else:
        if mass_gender.get("M", 0) > mass_gender.get("F", 0):
            prediction_gender = "Male"
        elif mass_gender.get("F", 0) > mass_gender.get("M", 0):
            prediction_gender = "Female"
        else:
            prediction_gender = "不確実"

    # Store results
    result_entry = {
        "User": user["user_id"],
        "Truth": user["ground_truth"],
        "Age_Pred": prediction_age,
    }
    for key, display_name in age_categories.items():
        result_entry[f"Age_Mass({display_name})"] = round(mass_age.get(key, 0.0), 3)
    result_entry["Age_Omega"] = round(mass_age.get("Omega", 0.0), 3)

    result_entry["Gender_Pred"] = prediction_gender
    for key, display_name in gender_categories.items():
        result_entry[f"Gender_Mass({display_name})"] = round(mass_gender.get(key, 0.0), 3)
    result_entry["Gender_Omega"] = round(mass_gender.get("Omega", 0.0), 3)
    result_entry["Reasoning"] = reasoning

    results.append(result_entry)

    time.sleep(6) # Pause to prevent API rate limit issues

# Display and Save Results
# ------------------------
df_results = pd.DataFrame(results)
display(df_results)

print("\n🎯 年齢と性別の推論結果が生成されました。\n")

df_results.to_excel("resultados_age_gender_llm_completo.xlsx", index=False)
print("💾 結果は 'resultados_age_gender_llm_completo.xlsx' に保存されました。")


✅ 35人のユーザーが正常にロードされました。
✅ ローカルデータの読み込み準備が完了しました。
推論を開始します...

ユーザー @poison_pain5618 (nan) を処理中...
    [警告] 画像が見つからないか、開けませんでした: u001.jpg
    [警告] 画像が見つからないか、開けませんでした: u001_4.jpg
    [警告] 画像が見つからないか、開けませんでした: u001_7.jpg
    [警告] 画像が見つからないか、開けませんでした: u001_9.jpg
    [警告] 画像が見つからないか、開けませんでした: u001_11.jpg
    [警告] 画像が見つからないか、開けませんでした: u001_14.jpg


In [ ]:
users_data[3]

{'user_id': '@1127_kumiko',
 'handle': 'mimu',
 'bio': '',
 'profile_picture_url': 'u004.jpg',
 'posts': [{'text': 'バイク決めてきた もう乗り方忘れてるが', 'image_file': None},
  {'text': 'バイク屋さんで貰ったシール\n一応メルカリで確認してみたら\nやはり出品されていた', 'image_file': None},
  {'text': '本日の激辛\n\n来来亭のマックス\nスープがスープじゃなく\nドロドロだったわ\n痛くは無いけどヘビー',
   'image_file': None},
  {'text': '今日、ラジオでね\r\nイナバヒロシだと思っていた\r\nって投稿が読まれていました。\r\n\r\n私はずっとコウジだと思っていました。\r\n\r\nぉひる\r\n腰のあるそーめんみたいな感じでした。',
   'image_file': 'u004_1.jpg'},
  {'text': '土曜日に友達とプチツー🏍️🏍️\r\n\r\n箕郷町の梅林を超えて\r\nカーブのキツイ山道を走ったせいか\r\n昨日から内腿が筋肉痛です。。\r\n\r\n今日、明日は天気良いから\r\nバイク通勤\r\n\r\n朝は寒くてなかなかエンジンがかからなくて焦る\r\n\r\n上信自動車道\r\n開通したので通ってみたよ\r\n2車線になって渋滞もなく\r\nいい塩梅だわ\r',
   'image_file': 'u004_2.jpg'},
  {'text': 'うちの会社は今日\r\nカレーの日🍛\r\n\r\nなんだけど\r\nカレーの匂いがわからない。。\r\n',
   'image_file': None},
  {'text': '小粒のさつまいも10キロ\r\n実家と娘宅とわけます\r\n\r\n甘すぎない甘さで\r\n美味しい🥰\r\n\r\n今日のおやつも\r\nふかし芋🍠\r\n',
   'image_file': 'u004_7.jpg'},
  {'text': '松ぼっくりに火だわ！\r\n\r\n昔の昼ドラ牡丹と薔薇が好きでちょくちょく観ている次女が間違えて使っている言葉である